# Timed Gel Swelling Inspection Loop

This cell starts a background thread that repeats the same `plate_1` workflow every 10 minutes: check that Jubilee is empty, move `plate_1` to Jubilee, optionally dispense water, capture an image, move Jubilee to the xArm pickup clearance position, and return the plate to its original location. This is intended for gel swelling or shrinking inspection over repeated hydration/imaging cycles.

The thread uses `robot_operation_lock` so repeated cycles cannot overlap with each other. Manual hardware-control cells can still conflict unless you also acquire the same lock or stop the loop before running them.

In [ ]:
gel_inspection_stop_event = threading.Event()
gel_inspection_thread = None

def run_gel_inspection_cycle(
    *,
    plate_id: str = "plate_1",
    add_water: bool = True,
    water_volume_ml: float = 10,
    camera_index: int = 0,
    move_speed: float = 100,
) -> Path:
    with robot_operation_lock:
        plate_at_jubilee = plate_handler.get_plate_at_position("Jubilee")
        if plate_at_jubilee is not None:
            raise RuntimeError(
                f"Jubilee is occupied by {plate_at_jubilee}. "
                "Check plate_handler.get_status() before starting the cycle."
            )

        original_position = plate_handler.get_plate_position(plate_id)
        image_path = None

        try:
            plate_handler.move_plate(
                plate_id=plate_id,
                destination_position_id="Jubilee",
                move_speed=move_speed,
            )

            if add_water:
                jubilee.pickup_tool(syringe_tool)
                jubilee.move_to(x=170, y=65, z=50, wait=True)
                wait_for_jubilee(jubilee)
                syringe_tool._dispense(water_volume_ml)
                wait_for_jubilee(jubilee)
                jubilee.park_tool()

            jubilee.pickup_tool(camera_tool)
            jubilee.move_to(x=170, y=106, z=75, wait=True)
            wait_for_jubilee(jubilee)
            image_path = capture_image(camera_index=camera_index)
            jubilee.park_tool()

            jubilee.move_to(x=300, y=10, z=240, wait=True)
            wait_for_jubilee(jubilee)

        finally:
            if plate_handler.get_plate_position(plate_id, required=False) == "Jubilee":
                jubilee.move_to(x=300, y=10, z=240, wait=True)
                wait_for_jubilee(jubilee)
                plate_handler.move_plate(
                    plate_id=plate_id,
                    destination_position_id=original_position,
                    move_speed=move_speed,
                )

        return image_path

def _gel_inspection_loop(
    *,
    interval_seconds: float = 10 * 60,
    add_water: bool = True,
    water_volume_ml: float = 10,
    camera_index: int = 0,
) -> None:
    cycle_number = 1
    while not gel_inspection_stop_event.is_set():
        try:
            print(f"Starting gel inspection cycle {cycle_number}: {datetime.now().isoformat(timespec='seconds')}")
            image_path = run_gel_inspection_cycle(
                add_water=add_water,
                water_volume_ml=water_volume_ml,
                camera_index=camera_index,
            )
            print(f"Finished cycle {cycle_number}; image: {image_path}")
        except Exception as exc:
            print(f"Gel inspection loop stopped after error: {exc}")
            gel_inspection_stop_event.set()
            break

        cycle_number += 1
        gel_inspection_stop_event.wait(interval_seconds)

def start_gel_inspection_loop(
    *,
    interval_seconds: float = 10 * 60,
    add_water: bool = True,
    water_volume_ml: float = 10,
    camera_index: int = 0,
) -> threading.Thread:
    global gel_inspection_thread

    if gel_inspection_thread is not None and gel_inspection_thread.is_alive():
        raise RuntimeError("Gel inspection loop is already running.")

    gel_inspection_stop_event.clear()
    gel_inspection_thread = threading.Thread(
        target=_gel_inspection_loop,
        kwargs={
            "interval_seconds": interval_seconds,
            "add_water": add_water,
            "water_volume_ml": water_volume_ml,
            "camera_index": camera_index,
        },
        daemon=True,
    )
    gel_inspection_thread.start()
    return gel_inspection_thread

def stop_gel_inspection_loop() -> None:
    gel_inspection_stop_event.set()

# Starts immediately, then repeats every 10 minutes. Set add_water=False for imaging-only cycles.
start_gel_inspection_loop(interval_seconds=10 * 60, add_water=True, water_volume_ml=10, camera_index=0)